# Oracle Security Lab: Automated DevSecOps Version
### Autonomous Defense-in-Depth & Shift-Left Security

This notebook is designed to run against the **Automated Laboratory Stack** (Port 1522). In this version, the database is "Born Secure"—initialization, user creation, and encryption wallets are handled automatically by Infrastructure-as-Code scripts.

---

## 0. Autonomous Verification
Unlike the manual lab, this environment initializes itself. Let's verify that the **Auto-Login Wallet** and **Resiliency** features are active.

In [29]:
# Verify that the Wallet is OPEN and using AUTOLOGIN
import subprocess
cmd = ['docker', 'exec', '-i', 'oracle-db-auto', '/bin/bash', '-c', "echo 'SELECT status, wallet_type FROM v$encryption_wallet;' | sqlplus -S / as sysdba"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)


STATUS			       WALLET_TYPE
------------------------------ --------------------
OPEN			       AUTOLOGIN
OPEN			       AUTOLOGIN
OPEN			       AUTOLOGIN




In [30]:
# Verify ARCHIVELOG mode (for ZDLRA resiliency)
import subprocess
cmd = ['docker', 'exec', '-i', 'oracle-db-auto', '/bin/bash', '-c', "echo 'SELECT log_mode FROM v$database;' | sqlplus -S / as sysdba"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)


LOG_MODE
------------
ARCHIVELOG




## 1. Reconnaissance
Scanning the automated environment to confirm system status.

In [31]:
!docker exec -i oracle-app-auto python lab/01_recon.py

Module 01: Reconnaissance

Database Banner: Oracle Database 21c Express Edition Release 21.0.0.0.0 - 
Production
          Instance Status           
┏━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Instance ┃ Host         ┃ Status ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━┩
│ XE       │ 128744d7ec01 │ OPEN   │
└──────────┴──────────────┴────────┘
              Security Options               
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Feature               ┃ Installed/Enabled ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Oracle Label Security │ FALSE             │
│ Oracle Database Vault │ FALSE             │
│ Unified Auditing      │ FALSE             │
└───────────────────────┴───────────────────┘


## 2. RBAC & Privilege Auditing
Auditing the pre-configured security baseline for "Shadow DBAs" and over-privileged users.

In [32]:
!docker exec -i oracle-app-auto python lab/02_rbac_audit.py

Module 02: RBAC Audit (Identity & Access Management)
  Users with DBA Role   
┏━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Grantee  ┃ Admin Opt ┃
┡━━━━━━━━━━╇━━━━━━━━━━━┩
│ IT_ADMIN │ NO        │
│ SYS      │ YES       │
│ SYSTEM   │ NO        │
└──────────┴───────────┘
                  Dangerous 'ANY' Privileges                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Grantee                    ┃ Privilege                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ APP_SUPPORT                │ CREATE ANY TABLE              │
│ APP_SUPPORT                │ DROP ANY TABLE                │
│ APP_SUPPORT                │ SELECT ANY TABLE              │
│ AQ_ADMINISTRATOR_ROLE      │ ENQUEUE ANY QUEUE             │
│ AQ_ADMINISTRATOR_ROLE      │ DEQUEUE ANY QUEUE             │
│ AQ_ADMINISTRATOR_ROLE      │ MANAGE ANY QUEUE              │
│ AUDSYS                     │ SELECT ANY DICTIONARY         │
│ CTXSYS                     │ INHERIT ANY PRIVILEGES 

## 3. Unified Auditing
Verifying that auditing policies were applied to sensitive PII data.

In [33]:
!docker exec -i oracle-app-auto python lab/03_audit_trail.py

Module 03: Unified Auditing
Audit policy 'AUDIT_PII_ACCESS' already exists.
Enabling audit policy...
Audit policy is active and enabled.
Simulating access to CORPORATE_PII.CLIENT_RECORDS...
Simulation: SELECT successful (500 records).
Simulation failed: ORA-00904: "CREATED_AT": invalid identifier
Help: https://docs.oracle.com/error-help/db/ora-00904/
Waiting for logs to process...
Flushing audit trail...
No audit logs found for AUDIT_PII_ACCESS.
Checking for ANY recent unified audit logs...
Recent Logs found (but not for our policy):
- 2024-09-30 01:20:18.891042 | SYS | ALTER PLUGGABLE DATABASE | Policy: 
ORA_SECURECONFIG
- 2024-09-30 01:17:51.824837 | SYS | CREATE LIBRARY | Policy: ORA_SECURECONFIG
- 2024-09-30 01:17:51.764149 | SYS | CREATE LIBRARY | Policy: ORA_SECURECONFIG
- 2024-09-30 01:17:41.831960 | SYS | ALTER PLUGGABLE DATABASE | Policy: 
ORA_SECURECONFIG


## 4. Data Redaction
Testing the real-time masking of PII (SSN and Credit Cards).

In [34]:
!docker exec -i oracle-app-auto python lab/04_data_masking.py

Module 04: Data Redaction
Redaction policy already exists.

Verifying Data Redaction as CORPORATE_PII user...
                     Redacted Data View                     
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name            ┃ SSN (Redacted) ┃ Credit Card (Partial) ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ Client Name 123 │                │ ************93-7      │
│ Client Name 124 │                │ ************49-7      │
│ Client Name 125 │                │ ************05-8      │
│ Client Name 126 │                │ ************61-9      │
│ Client Name 127 │                │ ************17-0      │
└─────────────────┴────────────────┴───────────────────────┘


## 5. SQL Injection & Secure Coding
Validating secure coding practices against the automated environment.

In [35]:
!docker exec -i oracle-app-auto python lab/05_sql_injection.py

Module 05: SQL Injection & Secure Coding

1. Standard Search (First182):
Executing Secure SQL: SELECT first_name, last_name, salary FROM HR.EMPLOYEES 
WHERE first_name = :name with bind :name='First182'
[('First182', 'Last182', 51820)]

2. Attempting SQL Injection on Vulnerable Function:
Attack Payload: Alice' OR '1'='1
Executing Vulnerable SQL: SELECT first_name, last_name, salary FROM HR.EMPLOYEES
WHERE first_name = 'Alice' OR '1'='1'
Result count (Expected 1, got all): 2000

3. Attempting SQL Injection on Secure Function:
Executing Secure SQL: SELECT first_name, last_name, salary FROM HR.EMPLOYEES 
WHERE first_name = :name with bind :name='Alice' OR '1'='1'
Result count (Correctly got 0): 0


## 6. Row-Level Security (VPD)
Testing multi-tenant data isolation.

In [36]:
!docker exec -i oracle-app-auto python lab/06_row_level_security.py

Module 06: Virtual Private Database (VPD)
Setting up VPD Policy...
VPD Policy 'DEPT_ACCESS_POLICY' is now active on HR.EMPLOYEES.

Simulating Session: Sales Manager (Authorized for Dept 1)
     Visible Records for Dept 1     
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┓
┃ First Name ┃ Last Name ┃ Dept ID ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━┩
│ First191   │ Last191   │ 1       │
│ First201   │ Last201   │ 1       │
│ First211   │ Last211   │ 1       │
│ First221   │ Last221   │ 1       │
│ First231   │ Last231   │ 1       │
└────────────┴───────────┴─────────┘
Total Rows Visible: 200 (Out of 2,000 total)

Simulating Session: Engineering Lead (Authorized for Dept 3)
     Visible Records for Dept 3     
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┓
┃ First Name ┃ Last Name ┃ Dept ID ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━┩
│ First183   │ Last183   │ 3       │
│ First193   │ Last193   │ 3       │
│ First203   │ Last203   │ 3       │
│ First213   │ Last213   │ 3       │
│ First223   │ Last223   │ 3       │
└

## 7. Transparent Data Encryption (TDE)
Confirming that storage-level encryption is active via the Auto-Login wallet.

In [37]:
!docker exec -i oracle-app-auto python lab/07_tde.py

Module 07: Transparent Data Encryption (TDE)
Checking current encryption status...
TDE is already active. SSN is encrypted with AES 192 bits key.


## 8. The Admin Trap (FGA)
Triggering the Fine-Grained Audit trail to catch high-privilege peeking.

In [38]:
!docker exec -i oracle-app-auto python lab/08_fga_trap.py

Module 08: Fine-Grained Auditing (FGA) - The 'Admin Trap'
Setting up Fine-Grained Auditing (FGA) 'Admin Trap'...
FGA Policy 'CATCH_DBA_PEEKING' is now active.
This policy monitors any non-owner access to sensitive PII columns.

Scenario: IT_ADMIN (DBA) is peeking at PII data...
Admin successfully retrieved data (Redaction might still be active, but FGA 
caught the attempt).
Waiting for audit trail flush (5s)...

Checking FGA Audit Logs (The Trap Result):
                          FGA Intrusion Detection Logs                          
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Timestamp           ┃ Intruder ┃ SQL Statement                               ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2026-04-24 09:33:36 │ IT_ADMIN │ SELECT full_name, ssn FROM                  │
│                     │          │ CORPORATE_PII.CLIENT_RE...                  │
│ 2026-04-24 05:29:58 │ IT_ADMIN │ SELECT full_name, ss

## 9. Secure Backup & Resiliency
Final audit of the encrypted backup status.

In [39]:
!docker exec -i oracle-app-auto python lab/09_secure_backup.py

Module 09: Secure Backup & Resiliency (ZDLRA Simulation)
Verifying RMAN Secure Backup Configuration...
Database Log Mode: ARCHIVELOG
RMAN Config: ENCRYPTION FOR DATABASE = ON
RMAN Config: ENCRYPTION ALGORITHM = 'AES256'

Auditing Backup Encryption Status (ZDLRA Proof):
No recent backup pieces found in CDB Root.
